## 1. Installation & Imports

Legacy standalone notebook. For new end-to-end runs, use `bgp_unified_features.ipynb` as the canonical extractor.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install pandas numpy tqdm

In [ ]:
import os
import json
import time
import logging
import warnings
import numpy as np
import pandas as pd
import bgpkit
from datetime import datetime, timedelta, timezone
from pathlib import Path
from collections import defaultdict, Counter
from urllib.request import urlretrieve
from urllib.error import URLError, HTTPError
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger('bgp_stat_features')

print('All imports successful.')
print(f'Pandas {pd.__version__}, NumPy {np.__version__}')
print(f'BGPKIT parser available ✓')

## 2. Configuration

All parameters are configurable. Adjust the collector, date range, window size, and labeling strategy below.

In [ ]:
# ============================================================================
# CONFIGURATION - Modify these parameters as needed
# LEGACY NOTE: this notebook is kept for standalone statistical experiments.
# The canonical production path is bgp_unified_features.ipynb.
# ============================================================================

# --- Data source ---
COLLECTOR = 'rrc04'                    # RIPE RIS collector
START_DATE = '2025-11-07'              # Start date (inclusive)
END_DATE = '2025-11-08'                # End date (inclusive)

# --- Feature extraction ---
WINDOW_SIZE = '5min'                   # Time window for feature aggregation
                                       # Options: '1s', '30s', '1min', '5min', '15min'

# --- Labeling ---
LABEL_STRATEGY = 'majority'            # Options: 'majority', 'conservative', 'weighted'
WEIGHTED_THRESHOLD = 0.4               # Threshold for 'weighted' strategy


# --- Ego-network scope (must match graph features notebook) ---
TARGET_AS = '3352'                    # Target AS number
EGO_K_HOP = 2                         # Same k-hop as graph features

# --- Output ---
BASE_DIR = Path('/home/smotaali/First_Full_Paper//bgp_stat_features_results')
GRAPH_OUTPUT_DIR = Path('/home/smotaali/First_Full_Paper/bgp_graph_features_results/output')
OUTPUT_DIR = BASE_DIR / 'output'
MRT_DIR = BASE_DIR / 'mrt_files'
TEMP_DIR = BASE_DIR / 'temp'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MRT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# --- Rare AS threshold ---
RARE_AS_THRESHOLD = 3                  # ASes appearing fewer times than this are 'rare'

# --- Suffix for output files ---
SUFFIX = f"{COLLECTOR}_AS{TARGET_AS}_{EGO_K_HOP}hop_{START_DATE}_{END_DATE}"

print(f"Collector:     {COLLECTOR}")
print(f"Date range:    {START_DATE} to {END_DATE}")
print(f"Window size:   {WINDOW_SIZE}")
print(f"Label strategy:{LABEL_STRATEGY}")
print(f"Output dir:    {OUTPUT_DIR}")

## 3. Data Discovery & Download

Downloads BGP UPDATE MRT files from RIPE RIS for the configured collector and date range.
UPDATE files are published every 5 minutes with filenames like `updates.YYYYMMDD.HHMM.gz`.

In [ ]:
def generate_update_urls(collector: str, start_date: str, end_date: str):
    """
    Generate URLs for all UPDATE dump files in the given date range.
    RIPE RIS publishes updates every 5 minutes.
    """
    urls = []
    start_dt = datetime.strptime(start_date, '%Y-%m-%d')
    end_dt = datetime.strptime(end_date, '%Y-%m-%d')
    
    current = start_dt
    while current < end_dt:
        year_month = current.strftime('%Y.%m')
        filename = f"updates.{current.strftime('%Y%m%d.%H%M')}.gz"
        url = f"https://data.ris.ripe.net/{collector}/{year_month}/{filename}"
        urls.append((url, filename, current))
        current += timedelta(minutes=5)
    
    return urls


def download_mrt_files(urls, output_dir):
    """
    Download MRT files, skipping already downloaded ones.
    Returns list of local paths.
    """
    local_paths = []
    
    for url, filename, ts in tqdm(urls, desc='Downloading UPDATE files'):
        local_path = output_dir / filename
        
        if local_path.exists() and local_path.stat().st_size > 0:
            local_paths.append((local_path, ts))
            continue
        
        try:
            urlretrieve(url, local_path)
            local_paths.append((local_path, ts))
        except (URLError, HTTPError) as e:
            logger.warning(f"Failed to download {filename}: {e}")
    
    return local_paths


# Generate URLs and download
update_urls = generate_update_urls(COLLECTOR, START_DATE, END_DATE)
print(f"Total UPDATE files to download: {len(update_urls)}")
print(f"Time range: {update_urls[0][2]} to {update_urls[-1][2]}")

downloaded_files = download_mrt_files(update_urls, MRT_DIR)
print(f"\nDownloaded/available: {len(downloaded_files)} files")

## 4. MRT Parsing Functions

Parse UPDATE MRT files using `bgpdump -m` into structured records.

In [ ]:
def parse_as_path_clean(as_path_str: str) -> str:
    """
    Clean AS path: remove AS-SETs, deduplicate prepending, filter private ASNs.
    Returns space-separated string of valid public ASNs.
    """
    if not as_path_str:
        return ''
    
    tokens = as_path_str.split()
    deduped = []
    in_as_set = False
    
    for token in tokens:
        if '{' in token:
            in_as_set = True
        if in_as_set:
            if '}' in token:
                in_as_set = False
            continue
        if '}' in token:
            continue
        try:
            asn = int(token)
            # Filter private ASNs (RFC 6996, RFC 7300)
            if (64512 <= asn <= 65534) or (4200000000 <= asn <= 4294967294):
                continue
            if not deduped or asn != deduped[-1]:
                deduped.append(str(asn))
        except ValueError:
            continue
    
    return ' '.join(deduped)


def parse_update_file(file_path: Path) -> list:
    """
    Parse a single MRT UPDATE file using bgpkit.Parser.
    Returns list of record dicts.
    """
    records = []
    
    logger.info(f"  Parsing: {file_path.name}")
    t0 = time.time()
    
    try:
        parser = bgpkit.Parser(url=str(file_path))
        for elem in parser:
            ts = datetime.fromtimestamp(elem.timestamp, tz=timezone.utc)
            
            # Build community string
            communities = ''
            if elem.communities:
                communities = ' '.join(str(c) for c in elem.communities)
            
            record = {
                'timestamp': ts,
                'type': elem.elem_type,           # 'A' or 'W'
                'peer_ip': elem.peer_ip or '',
                'peer_as': str(elem.peer_asn) if elem.peer_asn else '',
                'prefix': elem.prefix or '',
                'as_path': elem.as_path or '',
                'origin': elem.origin or '',
                'next_hop': elem.next_hop or '',
                'local_pref': elem.local_pref if elem.local_pref is not None else '',
                'med': elem.med if elem.med is not None else '',
                'community': communities,
            }
            records.append(record)
    
    except Exception as e:
        logger.warning(f"Error parsing {file_path.name}: {e}")
    
    elapsed = time.time() - t0
    logger.info(f"  -> {len(records):,} records in {elapsed:.1f}s")
    return records


print('Defined: parse_as_path_clean(), parse_update_file()')
print('  Uses bgpkit.Parser (same as graph features notebook)')

## 5. Parse All UPDATE Files into DataFrame

Parse all downloaded MRT files and combine into a single DataFrame.
This is the equivalent of the graph notebook's RIB parsing step.

In [ ]:
# Parse all UPDATE files
all_records = []

for gz_path, ts in tqdm(downloaded_files, desc='Parsing UPDATE files'):
    records = parse_update_file(gz_path)
    all_records.extend(records)

# Build DataFrame
updates_df = pd.DataFrame(all_records)
updates_df['timestamp'] = pd.to_datetime(updates_df['timestamp'], utc=True)
updates_df = updates_df.sort_values('timestamp').reset_index(drop=True)

# Clean AS paths (same logic as graph notebook)
updates_df['as_path_clean'] = updates_df['as_path'].apply(parse_as_path_clean)

# Summary
n_ann = (updates_df['type'] == 'A').sum()
n_wd = (updates_df['type'] == 'W').sum()
print(f"\nTotal UPDATE records: {len(updates_df):,}")
print(f"  Announcements: {n_ann:,}")
print(f"  Withdrawals:   {n_wd:,}")
print(f"  Time range:    {updates_df['timestamp'].min()} to {updates_df['timestamp'].max()}")
print(f"  Unique peers:  {updates_df['peer_as'].nunique()}")
print(f"  Unique prefixes: {updates_df['prefix'].nunique():,}")

In [ ]:
# ============================================================================
# Filter UPDATEs to ego-network scope (same scope as graph features)
# ============================================================================

# Reuse the configured TARGET_AS and EGO_K_HOP from the main configuration cell.
# Do not override them here, or the statistical scope can drift from the graph scope.

# Load ego-network AS set from graph features output
node_csv = list(GRAPH_OUTPUT_DIR.glob(f'node_level_timeseries_{SUFFIX}.csv'))

if node_csv:
    node_df = pd.read_csv(node_csv[0])
    ego_asns = set(node_df['asn'].unique().astype(str))
    print(f"Loaded ego-network from: {node_csv[0].name}")
else:
    raise FileNotFoundError(
        "No node-level CSV found. Run the graph features notebook first."
    )

print(f"Ego-network ASes: {len(ego_asns)} (AS{TARGET_AS}, {EGO_K_HOP}-hop)")

# ── Filter logic ──
# Keep an UPDATE if:
#   (a) The origin AS (last AS in path) is in the ego-network, OR
#   (b) TARGET_AS appears anywhere in the AS path, OR
#   (c) For withdrawals (no AS path): the peer_as is in the ego-network

print(f"\nTotal updates before filtering: {len(updates_df):,}")

def get_origin_as(path_str):
    """Extract origin AS (last AS in cleaned path)."""
    if isinstance(path_str, str) and path_str:
        asns = path_str.split()
        return asns[-1] if asns else ''
    return ''

# Vectorized filtering for speed (avoid row-by-row apply)
# (a) Origin AS is in ego-network
updates_df['_origin_as'] = updates_df['as_path_clean'].apply(get_origin_as)
mask_origin = updates_df['_origin_as'].isin(ego_asns)

# (b) TARGET_AS appears in AS path
mask_target = updates_df['as_path_clean'].str.contains(
    rf'(?:^|\s){TARGET_AS}(?:\s|$)', regex=True, na=False
)

# (c) For withdrawals without AS path: peer is in ego-network
mask_peer_wd = (updates_df['type'] == 'W') & (updates_df['peer_as'].astype(str).isin(ego_asns))

# Combine: keep if ANY condition is true
mask = mask_origin | mask_target | mask_peer_wd

updates_df = updates_df[mask].drop(columns=['_origin_as']).reset_index(drop=True)

# Summary
n_total = mask.shape[0]
n_kept = len(updates_df)
n_ann = (updates_df['type'] == 'A').sum()
n_wd = (updates_df['type'] == 'W').sum()

print(f"Total updates after ego-network filter: {n_kept:,}")
print(f"  Kept by origin AS in ego:    {mask_origin.sum():,}")
print(f"  Kept by TARGET_AS in path:   {mask_target.sum():,}")
print(f"  Kept by peer (withdrawals):  {mask_peer_wd.sum():,}")
print(f"  Reduction: {(1 - n_kept/n_total):.1%}")
print(f"\n  Announcements: {n_ann:,}")
print(f"  Withdrawals:   {n_wd:,}")
print(f"  Unique prefixes: {updates_df['prefix'].nunique():,}")
print(f"  Unique peers: {updates_df['peer_as'].nunique()}")

## 6. Statistical Feature Extraction Function


In [ ]:
def calculate_edit_distance(path1, path2):
    """
    Calculate Levenshtein edit distance between two AS paths.
    Paths should be lists of ASN strings.
    """
    if not path1 or not path2:
        return 0
    
    # Convert to lists if needed
    if isinstance(path1, str):
        path1 = [a for a in path1.split() if a.isdigit()]
    if isinstance(path2, str):
        path2 = [a for a in path2.split() if a.isdigit()]
    
    if not path1 or not path2:
        return 0
    
    m, n = len(path1), len(path2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if path1[i-1] == path2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    
    return dp[m][n]


def extract_statistical_features(df_window, prev_prefixes=None):
    """
    Extract all statistical features from a single time window.
    
    Args:
        df_window: DataFrame with UPDATE records for this window
        prev_prefixes: set of prefixes seen in previous window (for new_prefixes)
    
    Returns:
        features: dict of feature_name -> value
        current_prefixes: set of prefixes in this window (for next window)
    """
    features = {}
    
    announcements = df_window[df_window['type'] == 'A']
    withdrawals = df_window[df_window['type'] == 'W']
    
    # ── VOLUME FEATURES ──
    features['announcements'] = len(announcements)
    features['withdrawals'] = len(withdrawals)
    features['total_updates'] = len(df_window)
    
    # Per-second rates
    window_seconds = max(1, (df_window['timestamp'].max() - df_window['timestamp'].min()).total_seconds())
    features['ann_rate'] = features['announcements'] / window_seconds
    features['wd_rate'] = features['withdrawals'] / window_seconds
    
    # Ratios
    features['wd_ann_ratio'] = features['withdrawals'] / features['announcements'] if features['announcements'] > 0 else 0
    features['ann_wd_ratio'] = features['announcements'] / features['withdrawals'] if features['withdrawals'] > 0 else 0
    
    # ── PREFIX FEATURES ──
    ann_prefixes = set(announcements['prefix'].dropna()) if not announcements.empty else set()
    wd_prefixes = set(withdrawals['prefix'].dropna()) if not withdrawals.empty else set()
    
    features['unique_prefixes_ann'] = len(ann_prefixes)
    features['unique_prefixes_wd'] = len(wd_prefixes)
    
    # New prefixes (not seen in previous window)
    if prev_prefixes is not None:
        features['new_prefixes'] = len(ann_prefixes - prev_prefixes)
    else:
        features['new_prefixes'] = 0
    
    current_prefixes = ann_prefixes | wd_prefixes
    
    # Duplicates
    if not announcements.empty:
        dup_cols = ['peer_ip', 'peer_as', 'prefix', 'as_path', 'origin', 'next_hop']
        dup_cols = [c for c in dup_cols if c in announcements.columns]
        counts = announcements.groupby(dup_cols).size()
        features['dups'] = int(sum(c - 1 for c in counts if c > 1))
    else:
        features['dups'] = 0
    
    # Flaps: prefix both withdrawn and announced in same window
    features['flaps'] = len(wd_prefixes & ann_prefixes)
    
    # ── ORIGIN FEATURES ──
    if not announcements.empty and 'origin' in announcements.columns:
        origin_counts = announcements['origin'].value_counts()
        features['origin_IGP'] = int(origin_counts.get('IGP', 0))
        features['origin_INCOMPLETE'] = int(origin_counts.get('INCOMPLETE', 0))
        
        # Origin changes: prefixes with multiple origin attributes
        prefix_origins = announcements.groupby('prefix')['origin'].nunique()
        features['origin_changes'] = int((prefix_origins > 1).sum())
    else:
        features['origin_IGP'] = 0
        features['origin_INCOMPLETE'] = 0
        features['origin_changes'] = 0
    
    # ── AS PATH FEATURES ──
    valid_paths = announcements[announcements['as_path_clean'].notna() & (announcements['as_path_clean'] != '')]
    
    if not valid_paths.empty:
        path_lengths = valid_paths['as_path_clean'].apply(
            lambda p: len(p.split()) if isinstance(p, str) else 0
        )
        
        features['as_path_avg'] = float(path_lengths.mean())
        features['as_path_max'] = int(path_lengths.max())
        features['as_path_std'] = float(path_lengths.std()) if len(path_lengths) > 1 else 0.0
        
        # Unique AS paths per prefix
        unique_paths_per_prefix = valid_paths.groupby('prefix')['as_path_clean'].nunique()
        features['unique_as_path_max'] = int(unique_paths_per_prefix.max())
        
        # Edit distances between consecutive paths per prefix
        edit_distances = []
        edit_distance_dict = defaultdict(list)
        
        for prefix, group in valid_paths.groupby('prefix'):
            if len(group) >= 2:
                sorted_group = group.sort_values('timestamp')
                prev_path = None
                for _, row in sorted_group.iterrows():
                    current_path = row['as_path_clean']
                    if prev_path is not None:
                        dist = calculate_edit_distance(prev_path, current_path)
                        edit_distances.append(dist)
                        edit_distance_dict[prefix].append(dist)
                    prev_path = current_path
        
        if edit_distances:
            features['edit_distance_avg'] = float(np.mean(edit_distances))
            features['edit_distance_max'] = int(max(edit_distances))
            
            # Edit distance distribution (0-6)
            ed_counter = Counter(edit_distances)
            for i in range(7):
                features[f'edit_distance_dict_{i}'] = ed_counter.get(i, 0)
            
            # Unique edit distance distribution (0-1)
            unique_ed = {}
            for prefix, dists in edit_distance_dict.items():
                for d in set(dists):
                    unique_ed[d] = unique_ed.get(d, 0) + 1
            for i in range(2):
                features[f'edit_distance_unique_dict_{i}'] = unique_ed.get(i, 0)
        else:
            features['edit_distance_avg'] = 0.0
            features['edit_distance_max'] = 0
            for i in range(7):
                features[f'edit_distance_dict_{i}'] = 0
            for i in range(2):
                features[f'edit_distance_unique_dict_{i}'] = 0
    else:
        features['as_path_avg'] = 0.0
        features['as_path_max'] = 0
        features['as_path_std'] = 0.0
        features['unique_as_path_max'] = 0
        features['edit_distance_avg'] = 0.0
        features['edit_distance_max'] = 0
        for i in range(7):
            features[f'edit_distance_dict_{i}'] = 0
        for i in range(2):
            features[f'edit_distance_unique_dict_{i}'] = 0
    
    # ── RARE AS FEATURES ──
    if not valid_paths.empty:
        all_asns = []
        for path in valid_paths['as_path_clean']:
            if isinstance(path, str) and path:
                all_asns.extend(path.split())
        
        asn_counts = Counter(all_asns)
        rare_asns = [a for a, c in asn_counts.items() if c < RARE_AS_THRESHOLD]
        features['number_rare_ases'] = len(rare_asns)
        features['rare_ases_ratio'] = len(rare_asns) / len(all_asns) if all_asns else 0.0
    else:
        features['number_rare_ases'] = 0
        features['rare_ases_ratio'] = 0.0
    
    # ── IMPLICIT WITHDRAWAL FEATURES ──
    if not announcements.empty:
        imp_wd = 0
        imp_wd_spath = 0
        imp_wd_dpath = 0
        
        for (prefix, peer), group in announcements.groupby(['prefix', 'peer_ip']):
            if len(group) > 1:
                imp_wd += 1
                if 'as_path_clean' in group.columns:
                    if group['as_path_clean'].nunique() == 1:
                        imp_wd_spath += 1
                    else:
                        imp_wd_dpath += 1
        
        features['imp_wd'] = imp_wd
        features['imp_wd_spath'] = imp_wd_spath
        features['imp_wd_dpath'] = imp_wd_dpath
    else:
        features['imp_wd'] = 0
        features['imp_wd_spath'] = 0
        features['imp_wd_dpath'] = 0
    
    # ── BEHAVIORAL FEATURES ──
    # NADAS: basic attack pattern heuristic
    specific_prefixes = sum(1 for p in df_window['prefix'].dropna()
                           if isinstance(p, str) and '/32' in p)
    features['nadas'] = specific_prefixes + (10 if features['wd_ann_ratio'] > 0.5 else 0)
    
    # Unique active peers
    features['unique_peers'] = df_window['peer_as'].nunique()
    
    return features, current_prefixes


print('Defined: extract_statistical_features(df_window, prev_prefixes)')
print(f'  Extracts 37 statistical features per time window')

## 7. Per-Window Processing Loop

Split the UPDATE data into time windows and extract features for each window.
This is analogous to the graph notebook's per-snapshot processing loop.

In [ ]:
# ============================================================================
# Process each time window
# ============================================================================

print(f"Window size: {WINDOW_SIZE}")
print(f"Processing...\n")

# Create time windows using pandas Grouper
updates_df = updates_df.set_index('timestamp')
grouped = updates_df.groupby(pd.Grouper(freq=WINDOW_SIZE))

feature_rows = []
prev_prefixes = None
n_windows = len(grouped)

for window_start, window_df in tqdm(grouped, total=n_windows, desc='Extracting features'):
    if window_df.empty:
        continue
    
    window_df_reset = window_df.reset_index()
    window_end = window_start + pd.Timedelta(WINDOW_SIZE)
    
    try:
        features, current_prefixes = extract_statistical_features(window_df_reset, prev_prefixes)
        
        # Add metadata
        features['window_start'] = window_start
        features['window_end'] = window_end
        features['collector'] = COLLECTOR
        features['window_size'] = WINDOW_SIZE
        
        feature_rows.append(features)
        prev_prefixes = current_prefixes
        
    except Exception as e:
        logger.warning(f"Error in window {window_start}: {e}")
        import traceback; traceback.print_exc()

# Reset index back
updates_df = updates_df.reset_index()

print(f"\nProcessed {len(feature_rows)} windows successfully")

## 8. Combine Results into Time-Series DataFrame

In [ ]:
# ============================================================================
# Combine per-window results into time-series DataFrame
# ============================================================================

if len(feature_rows) == 0:
    raise RuntimeError("No successful windows were produced.")

stat_ts_df = pd.DataFrame(feature_rows)
meta_cols = ['window_start', 'window_end', 'collector', 'window_size']
feature_cols = [c for c in stat_ts_df.columns if c not in meta_cols]
stat_ts_df = stat_ts_df[meta_cols + sorted(feature_cols)]
stat_ts_df['window_start'] = pd.to_datetime(stat_ts_df['window_start'])
stat_ts_df['window_end'] = pd.to_datetime(stat_ts_df['window_end'])
stat_ts_df = stat_ts_df.sort_values('window_start').reset_index(drop=True)

print(f"Statistical feature time series: {stat_ts_df.shape}")
print(f"  Windows: {len(stat_ts_df)}")
print(f"  Features: {len(feature_cols)}")
print(f"  Time range: {stat_ts_df['window_start'].min()} to {stat_ts_df['window_end'].max()}")
print(f"\nFeature columns:")
for c in sorted(feature_cols):
    print(f"  {c}")

## 9. Export Results

In [ ]:
# ============================================================================
# Filter to keep only the 15 selected features (post-redundancy-analysis)
# ============================================================================

STAT_META_COLS = ['window_start', 'window_end', 'collector', 'window_size']

KEEP_STATISTICAL = [
    # Volume (3)
    'announcements', 'withdrawals', 'wd_ann_ratio',
    # Prefix (3)
    'unique_prefixes_ann', 'new_prefixes', 'flaps',
    # Origin (1)
    'origin_changes',
    # AS Path (5)
    'as_path_avg', 'as_path_max', 'unique_as_path_max',
    'edit_distance_avg', 'edit_distance_max',
    # Rare AS (1)
    'number_rare_ases',
    # Implicit WD (1)
    'imp_wd_dpath',
    # Behavioral (1)
    'unique_peers',
]

DROP_STATISTICAL = [
    'total_updates',       # = announcements + withdrawals
    'ann_rate',            # = announcements / window
    'wd_rate',             # = withdrawals / window
    'ann_wd_ratio',        # = 1 / wd_ann_ratio
    'unique_prefixes_wd',  # redundant with withdrawals
    'dups',                # redundant with announcements (VIF=66.5)
    'origin_IGP',          # redundant with announcements (r=0.99)
    'origin_INCOMPLETE',   # correlated with main cluster (VIF=269M)
    'as_path_std',         # redundant with as_path_max
    'edit_distance_dict_0', 'edit_distance_dict_1', 'edit_distance_dict_2',
    'edit_distance_dict_3', 'edit_distance_dict_4', 'edit_distance_dict_5',
    'edit_distance_dict_6',
    'edit_distance_unique_dict_0', 'edit_distance_unique_dict_1',
    'rare_ases_ratio',     # redundant with volume
    'imp_wd',              # redundant with announcements (r=0.98)
    'imp_wd_spath',        # benign signal
    'nadas',               # weak heuristic
]

# Build selected DataFrame
available_sel = [c for c in KEEP_STATISTICAL if c in stat_ts_df.columns]
missing_sel = [c for c in KEEP_STATISTICAL if c not in stat_ts_df.columns]
stat_ts_df_sel = stat_ts_df[STAT_META_COLS + sorted(available_sel)]

print(f"Statistical features: {stat_ts_df.shape[1]} cols → {stat_ts_df_sel.shape[1]} cols "
      f"({len(available_sel)} features + {len(STAT_META_COLS)} meta)")
if missing_sel:
    print(f"  ⚠ Missing features: {missing_sel}")

# Verify drops
actually_dropped = [c for c in DROP_STATISTICAL if c in stat_ts_df.columns]
print(f"  Dropped: {len(actually_dropped)} features")
print(f"\n✓ Selected {len(available_sel)} / 15 statistical features")

In [ ]:
# ============================================================================
# Export
# ============================================================================

# 1. Full statistical features
full_path = OUTPUT_DIR / f"stat_features_full_{SUFFIX}.csv"
stat_ts_df.to_csv(full_path, index=False)
print(f"Full statistical features: {full_path}  ({stat_ts_df.shape[1]} cols)")

# 2. Summary statistics
summary = stat_ts_df[sorted(feature_cols)].describe().T
summary_path = OUTPUT_DIR / f"stat_features_summary_{SUFFIX}.csv"
summary.to_csv(summary_path)
print(f"Summary statistics:        {summary_path}")

# 3. Processing report
report = {
    'collector': COLLECTOR,
    'start_date': START_DATE,
    'end_date': END_DATE,
    'window_size': WINDOW_SIZE,
    'total_updates': len(updates_df),
    'total_windows': len(stat_ts_df),
    'n_features': len(feature_cols),
    'features': sorted(feature_cols),
    'created_at': datetime.now(timezone.utc).isoformat(),
}

# ── 4. Selected statistical features ──
sel_path = OUTPUT_DIR / f"stat_features_SELECTED_{SUFFIX}.csv"
stat_ts_df_sel.to_csv(sel_path, index=False)
print(f"Selected stat features: {sel_path}  ({stat_ts_df_sel.shape[1]} cols)")

# ── 5. Selected summary ──
sel_feature_cols = [c for c in stat_ts_df_sel.columns if c not in STAT_META_COLS]
sel_summary = stat_ts_df_sel[sorted(sel_feature_cols)].describe().T
sel_summary_path = OUTPUT_DIR / f"stat_features_SELECTED_summary_{SUFFIX}.csv"
sel_summary.to_csv(sel_summary_path)
print(f"Selected summary:      {sel_summary_path}")

# ── 6. Update report with selection info ──
report['selected_features'] = sorted(available_sel)
report['dropped_features'] = sorted(actually_dropped)
report['n_selected'] = len(available_sel)
report['n_dropped'] = len(actually_dropped)


print(f"\n{'='*60}")
print(f"OUTPUT FILES:")
print(f"{'='*60}")
print(f"  Full ({stat_ts_df.shape[1]} cols):     {full_path.name}")
print(f"  Selected ({stat_ts_df_sel.shape[1]} cols): {sel_path.name}")


report_path = OUTPUT_DIR / f"stat_features_report_{SUFFIX}.json"
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)
print(f"Processing report:         {report_path}")

print(f"\n{'='*60}")
print(f"ALL EXPORTS COMPLETE")
print(f"{'='*60}")

## 10. Quick Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Plot key features over time
key_features = ['announcements', 'withdrawals', 'flaps', 'edit_distance_avg',
                'as_path_avg', 'unique_prefixes_ann', 'origin_changes', 'number_rare_ases']

available_keys = [f for f in key_features if f in stat_ts_df.columns]

n_plots = len(available_keys)
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 3 * n_plots), sharex=True)
if n_plots == 1:
    axes = [axes]

for ax, feat in zip(axes, available_keys):
    ax.plot(stat_ts_df['window_start'], stat_ts_df[feat], linewidth=0.8)
    ax.set_ylabel(feat, fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[-1].set_xlabel('Time (UTC)')
fig.suptitle(f'BGP Statistical Features — {COLLECTOR} ({START_DATE} to {END_DATE}, window={WINDOW_SIZE})',
             fontsize=13, fontweight='bold')
plt.tight_layout()

fig_path = OUTPUT_DIR / f"stat_features_timeseries_{SUFFIX}.png"
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Saved: {fig_path}")
plt.show()

In [ ]:
# ============================================================================
# ENRICH TOPOLOGY HTML WITH STATISTICAL FEATURES
# ============================================================================
# Finds HTML files generated by bgp_graph_features.ipynb and injects
# statistical features into the DATA blob so the "Activity" tab appears.
# ============================================================================

import re
import json as _json
from pathlib import Path

# ── Where graph notebook wrote its HTML files ──
GRAPH_VIZ_DIR = Path(f"/home/smotaali/First_Full_Paper/bgp_graph_features_results/output/visualization/target_{TARGET_AS}")

# ── Where this notebook wrote its stat CSV ──
# (adjust if your stat output path is different)
STAT_CSV = OUTPUT_DIR / f"stat_features_full_{SUFFIX}.csv"

# Features to inject
STAT_FEATURES_FOR_VIZ = [
    'announcements', 'withdrawals', 'wd_ann_ratio',
    'unique_prefixes_ann', 'new_prefixes', 'flaps',
    'origin_changes', 'as_path_avg', 'as_path_max',
    'edit_distance_avg', 'edit_distance_max',
    'number_rare_ases', 'imp_wd_dpath', 'unique_peers',
]

if not GRAPH_VIZ_DIR.exists():
    print(f"No visualization directory found at {GRAPH_VIZ_DIR}")
    print("Run bgp_graph_features.ipynb first to generate HTML topology files.")
elif not STAT_CSV.exists():
    print(f"No stat CSV found at {STAT_CSV}")
else:
    # Load stat features
    stat_df = pd.read_csv(STAT_CSV)
    stat_df['window_start'] = pd.to_datetime(stat_df['window_start'])
    print(f"Loaded {len(stat_df)} stat windows from {STAT_CSV.name}")

    # Pattern to extract DATA JSON from HTML
    DATA_PATTERN = re.compile(r'const DATA\s*=\s*(\{.*?\});\s*\n', re.DOTALL)

    html_files = sorted(GRAPH_VIZ_DIR.glob("topology_*.html"))
    n_enriched = 0
    n_skipped = 0

    for html_path in html_files:
        try:
            html_content = html_path.read_text(encoding='utf-8')
            match = DATA_PATTERN.search(html_content)
            if not match:
                print(f"  SKIP {html_path.name}: could not find DATA JSON")
                n_skipped += 1
                continue

            # Parse existing DATA
            data_json_str = match.group(1)
            data_blob = _json.loads(data_json_str)

            # Find nearest stat window for this snapshot
            snap_ts = pd.to_datetime(data_blob.get('snapshot_ts', ''))
            if pd.isna(snap_ts):
                n_skipped += 1
                continue

            time_diffs = (stat_df['window_start'] - snap_ts).abs()
            nearest_idx = time_diffs.idxmin()

            # Only match if within 10 minutes
            if time_diffs[nearest_idx].total_seconds() > 600:
                print(f"  SKIP {html_path.name}: no stat window within 10min of {snap_ts}")
                n_skipped += 1
                continue

            nearest_row = stat_df.loc[nearest_idx]
            stat_dict = {}
            for feat in STAT_FEATURES_FOR_VIZ:
                if feat in nearest_row.index:
                    v = nearest_row[feat]
                    stat_dict[feat] = round(float(v), 6) if pd.notna(v) else None
            
            # Inject into data blob
            data_blob['stat_features'] = stat_dict

            # Replace in HTML
            new_json_str = _json.dumps(data_blob, separators=(',', ':'))
            new_html = html_content[:match.start(1)] + new_json_str + html_content[match.end(1):]

            html_path.write_text(new_html, encoding='utf-8')
            n_enriched += 1

        except Exception as e:
            print(f"  ERROR {html_path.name}: {e}")
            n_skipped += 1

    print(f"\nDone: {n_enriched} HTML files enriched, {n_skipped} skipped")
    print(f"Open any topology HTML → click target AS → 'Activity' tab now shows stat features")

## Feature Definitions Reference

### Volume Features
| Feature | Description |
|---------|-------------|
| `announcements` | Number of BGP route announcements (type A) |
| `withdrawals` | Number of BGP route withdrawals (type W) |
| `total_updates` | Total BGP UPDATE messages |
| `ann_rate` | Announcements per second |
| `wd_rate` | Withdrawals per second |
| `wd_ann_ratio` | Ratio of withdrawals to announcements |
| `ann_wd_ratio` | Ratio of announcements to withdrawals |

### Prefix Features
| Feature | Description |
|---------|-------------|
| `unique_prefixes_ann` | Number of distinct prefixes announced |
| `unique_prefixes_wd` | Number of distinct prefixes withdrawn |
| `new_prefixes` | Prefixes not seen in the previous time window |
| `dups` | Duplicate announcements (same prefix+path+attributes) |
| `flaps` | Prefixes both withdrawn and re-announced in same window |

### Origin Features
| Feature | Description |
|---------|-------------|
| `origin_IGP` | Announcements with IGP origin attribute |
| `origin_INCOMPLETE` | Announcements with INCOMPLETE origin attribute |
| `origin_changes` | Prefixes with multiple different origin attributes |

### AS Path Features
| Feature | Description |
|---------|-------------|
| `as_path_avg` | Mean AS path length (after prepending removal) |
| `as_path_max` | Maximum AS path length |
| `as_path_std` | Standard deviation of AS path lengths |
| `unique_as_path_max` | Max number of unique paths to any single prefix |
| `edit_distance_avg` | Mean edit distance between consecutive paths per prefix |
| `edit_distance_max` | Maximum edit distance observed |
| `edit_distance_dict_0..6` | Distribution of edit distances (counts for 0-6) |
| `edit_distance_unique_dict_0..1` | Unique edit distance distribution |

### Rare AS Features
| Feature | Description |
|---------|-------------|
| `number_rare_ases` | Count of ASes appearing fewer than threshold times |
| `rare_ases_ratio` | Fraction of rare ASes in all observed ASes |

### Implicit Withdrawal Features
| Feature | Description |
|---------|-------------|
| `imp_wd` | Prefix-peer pairs with multiple announcements (implicit WD) |
| `imp_wd_spath` | Implicit WDs with same AS path |
| `imp_wd_dpath` | Implicit WDs with different AS path |

### Behavioral Features
| Feature | Description |
|---------|-------------|
| `nadas` | Attack pattern heuristic score |
| `unique_peers` | Number of distinct BGP peers active in window |